# Tarea 2 · OCR en Google Colab

La laptop de trabajo, con 7,7 GB de RAM y sin GPU, no alcanzaba para PaddleOCR. El sistema se quedaba sin memoria y cada página tomaba entre 3 y 9 minutos. Por eso el OCR de los documentos pendientes se corrió aquí, con **la misma versión y la misma configuración** que en `01_extraccion.ipynb`.

1. Baja las imágenes desde Wikimedia Commons con `descargar.py` y verifica con SHA-256 que sean idénticas a las del corpus local.
2. Corre PaddleOCR y guarda el JSON crudo (textos, confianzas y cajas) y la imagen con las cajas de cada documento.
3. Descarga `ocr_resultados.zip`, que se copia a `datos/raw/documentos/ocr/`. El notebook 01 lee esos JSON y no vuelve a correr el OCR.

Este notebook se ejecutó en Colab el 23 de octubre de 2026. Las salidas de esa ejecución no quedaron en esta copia, pero sus resultados son los JSON que usa el notebook 01, y la sección 2 de ese notebook explica el proceso.

In [ ]:
# 1. Entorno con PaddleOCR 3.7.0 sobre PaddlePaddle 3.2.2 (GPU si hay, CPU si no)
import subprocess
gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if gpu:
    !pip -q install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
else:
    !pip -q install paddlepaddle==3.2.2
!pip -q install paddleocr==3.7.0
# El torch de Colab choca con la librería NCCL que trae paddlepaddle-gpu y PaddleOCR no lo necesita
!pip -q uninstall -y torch torchvision torchaudio
import glob, shutil, site  # pip deja restos de torch que modelscope (dependencia de PaddleX) intenta importar
for resto in glob.glob(site.getsitepackages()[0] + "/torch*"):
    shutil.rmtree(resto, ignore_errors=True)
print("GPU:", gpu)

In [ ]:
# 2. Imágenes desde Wikimedia Commons, con el mismo descargar.py del repo
import hashlib, json, sys, urllib.request
from pathlib import Path
from PIL import Image
urllib.request.urlretrieve("https://raw.githubusercontent.com/nicosil02/Python-Intermedio/parte-1-corpus/tarea_2/descargar.py", "descargar.py")
sys.path.insert(0, ".")
import descargar as d

ESPERADO = {
 "doc_006": "ec659d772f123102e8cfb54930b5c3e17df6fc5bea289b5b918d07f58b59b0c4",
 "doc_007": "6f64ee63f3e047c34f8b266d3e8cec63edbfa0888bfda41d2aaa652066e3ab56",
 "doc_008": "c36b03645401a4de83383a76b54f36dc2d0a211c58ff944ea02e8fff9527fda4",
 "doc_009": "b14555f8d9d3dc094f153489b6ac11866fc9b41c793b45c6fb4810e379a14788",
 "doc_010": "03dd165ae137cfa8845f4f29a2ae6510ddc34949d30c271f498ad359b67b8ae1",
 "doc_011": "938d58596365d7395ab2238e2ea1bb4732d040fe173ae6f9f9be9e781315cf35",
 "doc_012": "0c54bfa0758c5876f9b86c833417cb4dc6ed0bfead298ec13d7277187b2d566a",
 "doc_013": "361d6b4b4a0370af5d9fb59c7a7c722aec7c827a07d47f8920f3de23e7a213ad",
 "doc_014": "3a1b34dbd9e5cf4015fd8841692436a330644571ff82f371ad8f660f63b1fbe4",
 "doc_015": "15924be41979056bdda4f1a3c70d9081967474781b1b4e7e8ce087a9f2dbf63f",
 "doc_016": "df7110e1894701c8ec5093c2b1cc92a0ba309a6d8cde928c78e489a5eecf8c90",
 "doc_017": "03dd2dba2165f3c3ea416a6bf336c4579c07703916449c9a4ff50d4c307ce9c4",
 "doc_018": "250ccfd1dc8534b15f0cf01a0e9603f7252155340754c54c08272cf68b5e4168",
 "doc_019": "c856cfe49879fa19e12846198f8bc747c09acd083863fab5983587f3dc5a3a6a",
 "doc_020": "5e580170c0cc938b8cb7c6909018e49413aedcbefbf7ec6676ba7f324d00b27b",
 "doc_021": "6d2eaea032c9ada1802941fb6d55c558f8181c2e3cbd15264c812b5f3bda6576",
 "doc_022": "fa63b411cc39cbef873c345c6e5583bed142566a619e53522ce7a1b2e84b4e42",
 "doc_023": "a0f0433a72a49e0ef94329bb294e9aa2fe3a399a21eb2115ebd48ae9c3679001",
 "doc_024": "37ed6a204727748b5d18179390cf353b6792e0bb956120c258ef4d67f4348cef",
 "doc_025": "89e6214ad92ed26ebb59f944e2bf24e0528395d9b0e43c305beca826bcd6e70a",
 "doc_r01": "2d46b331e477710c162c74fd3abd951086591828deb7730ae8fdfe83170170c6",
 "doc_r02": "9807485c69c489202fcabb27f87045760c47740ab43211836642b04d4b4a4be7",
 "doc_r03": "210df30efb259b2ce75526b0b4103045e3a9f7f15c4da3ef2e6fbdbbabdef465",
 "doc_r04": "9862c1de60920b3c53e7ac44f71622349cfdab9b06a22510e566be6267a817df"
}
carpeta = Path("documentos"); carpeta.mkdir(exist_ok=True)
for id_, (tipo, titulo, pagina, recorte) in {**d.DOCUMENTOS, **d.RESERVAS}.items():
    if id_ not in ESPERADO:
        continue
    destino = carpeta / f"{id_}.jpg"
    d.bajar(d.ficha(titulo, pagina)["url"], destino)
    if recorte:
        im = Image.open(destino)
        im.crop(tuple(int(v * t) for v, t in zip(recorte, im.size * 2))).save(destino, quality=95)
    igual = hashlib.sha256(destino.read_bytes()).hexdigest() == ESPERADO[id_]
    print(id_, "idéntica a la local" if igual else "DISTINTA a la local")

In [ ]:
# 3. OCR con la configuración fija del corpus
import json, time, numpy as np, paddle, paddleocr
from paddleocr import PaddleOCR
CONFIG = dict(lang="es", use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
              text_detection_model_name="PP-OCRv5_mobile_det", text_det_limit_side_len=1280, text_det_limit_type="max")
print("PaddleOCR", paddleocr.__version__, "| PaddlePaddle", paddle.__version__, "| dispositivo", paddle.device.get_device())
ocr = PaddleOCR(**CONFIG)
salida = Path("ocr"); salida.mkdir(exist_ok=True)
for img in sorted(carpeta.glob("doc_*.jpg")):
    t0 = time.perf_counter()
    r = ocr.predict(str(img))[0]
    r.save_to_img(str(salida / f"{img.stem}_cajas.jpg"))
    res = {"rec_texts": list(r["rec_texts"]), "rec_scores": [float(s) for s in r["rec_scores"]],
           "rec_boxes": np.asarray(r["rec_boxes"]).tolist(), "segundos": time.perf_counter() - t0,
           "config": CONFIG, "entorno": f"Colab {paddle.device.get_device()}"}
    (salida / f"{img.stem}.json").write_text(json.dumps(res, ensure_ascii=False), encoding="utf-8")
    print(img.stem, len(res["rec_texts"]), "líneas", round(res["segundos"], 1), "s")

In [ ]:
# 4. Descargar los resultados
!cd ocr && zip -q -r ../ocr_resultados.zip .
from google.colab import files
files.download("ocr_resultados.zip")